实现策略1：在State中存储，存储在内存中

In [1]:
from typing import TypedDict, Annotated

# 假设消息合并函数 add_messages 已定义
def add_messages(old_list: list, new_list: list) -> list:
    return old_list + new_list

class ChatBotWithMemoryState(TypedDict):
    # 短期记忆
    messages: Annotated[list, add_messages]
    # 长期记忆
    user_name: str
    user_preferences: dict  # {"language": "zh", "tone": "formal"}
    learned_facts: list    # ["用户喜欢咖啡", "用户在北京"]


def personalized_chatbot(state: ChatBotWithMemoryState) -> dict:
    """个性化的聊天机器人"""
    messages = state["messages"]
    user_name = state.get("user_name", "朋友")
    preferences = state.get("user_preferences", {})
    facts = state.get("learned_facts", [])

    # 构建个性化的系统提示
    system_prompt = f"""
        你是一个友好的助手，正在与 {user_name} 对话。

        已知信息：
        {chr(10).join(f"- {fact}" for fact in facts)}

        用户偏好：
        {chr(10).join(f"- {k}: {v}" for k, v in preferences.items())}

        请根据这些信息提供个性化的回复。
        """
    full_messages = [{"role":"system","content":system_prompt}] + messages
    response = model.invoke(full_messages)

    return {"messages": [response]}

def extract_facts(state: ChatBotWithMemoryState) -> dict:
    """从对话中提取事实"""
    last_messages = state["messages"][-2:]  # 最近的一轮对话

    extraction_prompt = f"""
        从以下对话中提取关于用户的事实：
        {last_messages}

        返回 JSON 格式：
        {{
            "facts": ["fact1", "fact2"],
            "preferences": {{"key": "value"}}
        }}
        如果没有新信息，返回空列表/字典。
    """
    result = model.invoke([{"role":"user","content":extraction_prompt}])
    # 解析结果（简化处理）
    import json
    try:
        extracted = json.loads(result.content)
        current_facts = state.get("learned_facts", [])
        current_prefs = state.get("user_preferences", {})
        return {
            "learned_facts": current_facts + extracted.get("facts", []),
            "user_preferences": {**current_prefs, **extracted.get("preferences", {})}
        }
    except:
        return {}


实现策略2：外部存储（向量数据库）

In [2]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from typing import TypedDict, Annotated, List

# 假设 add_messages 已提前定义
def add_messages(old_list: list, new_list: list) -> list:
    return old_list + new_list

class ChatBotWithVectorMemory(TypedDict):
    messages: Annotated[list, add_messages]
    user_id: str

# 初始化向量数据库
embeddings = OpenAIEmbeddings()
vectorstore = Chroma(
    collection_name="user_memories",
    embedding_function=embeddings
)

def retrieve_relevant_memories(user_id: str, query: str, k: int = 3) -> list[str]:
    """从向量库检索相关记忆"""
    # 检索与当前查询相关的记忆
    results = vectorstore.similarity_search(
        query,
        k=k,
        filter={"user_id": user_id}
    )
    return [doc.page_content for doc in results]

def save_memory(user_id: str, memory: str):
    """保存新的记忆"""
    vectorstore.add_texts(
        texts=[memory],
        metadatas=[{"user_id": user_id}]
    )

def chatbot_with_vector_memory(state: ChatBotWithVectorMemory) -> dict:
    """使用向量数据库的聊天机器人"""
    messages = state["messages"]
    user_id = state["user_id"]
    last_message = messages[-1].content

    # 检索相关记忆
    relevant_memories = retrieve_relevant_memories(user_id, last_message)

    # 构建提示
    system_prompt = f"""
        你是一个友好的助手。

        关于这个用户，你之前学到的相关信息：
        {chr(10).join(f"- {mem}" for mem in relevant_memories)}

        利用这些信息提供个性化的回复。
        """

    full_messages = [{"role":"system","content":system_prompt}] + messages
    response = model.invoke(full_messages)

    # 保存新的记忆（可选）
    # save_memory(user_id, f"用户说: {last_message}，我回复: {response.content}")
    return {"messages": [response]}

/var/folders/nd/wxdc_5b13hj3k9jdg1w8gd8r0000gn/T/ipykernel_26329/534858966.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.